# 모두몰 매출 TOP 리포트 — JOIN 분석 (customers, orders, products, order_items)

**과제**: 여러 테이블을 이어야 답할 수 있는 분석 질문 5개 + SQL(JOIN) + 인사이트

**환경 안내**: BigQuery 문법 대신 로컬 **DuckDB**로 동일 스키마/데이터를 구성해 실행했습니다.
문법은 BigQuery Standard SQL과 거의 동일합니다.

**스키마 갱신**: 이번엔 `order_items`(order_id, product_id, quantity, unit_price)가 추가되어
`orders ↔ order_items ↔ products`가 전부 연결됩니다. 그래서 "카테고리별 베스트셀러" 같은
질문도 만들 수 있게 되었습니다.

**조건 체크리스트**
- INNER JOIN 최소 1회 → Q1, Q2, Q3, Q5
- LEFT JOIN 최소 1회 → Q4
- 서브쿼리(WHERE 또는 FROM) 최소 1회 → Q3(FROM+상관 서브쿼리), Q5(WHERE)
- JOIN + GROUP BY 집계 최소 1개 → Q1, Q2, Q3


## 0. 환경 설정 및 테이블 생성 (DuckDB)

In [1]:
import duckdb
import pandas as pd

con = duckdb.connect(database=":memory:")

con.execute("""
CREATE OR REPLACE TABLE customers (
    customer_id STRING,
    name STRING,
    country STRING,
    signup_date DATE,
    grade STRING
);
""")
con.execute("""
INSERT INTO customers VALUES
  ('C001', '김민준', 'Korea', '2023-01-05', 'Gold'),
  ('C002', '이서연', 'Korea', '2023-02-11', 'Silver'),
  ('C003', '박도윤', 'Japan', '2023-02-20', 'Bronze'),
  ('C004', '최지우', 'USA', '2023-03-03', 'Gold'),
  ('C005', '정하준', 'Korea', '2023-03-15', 'Silver'),
  ('C006', '강서윤', 'Korea', '2023-04-01', 'Bronze'),
  ('C007', '조은우', 'Japan', '2023-04-18', 'Silver'),
  ('C008', '윤지호', 'USA', '2023-05-09', 'Gold'),
  ('C009', '임하은', 'Korea', '2023-05-22', 'Bronze'),
  ('C010', '한예준', 'Korea', '2023-06-02', 'Silver'),
  ('C011', '오시우', NULL, '2023-06-19', 'Bronze'),
  ('C012', '신아린', 'Japan', '2023-07-07', 'Silver'),
  ('C013', '권준서', 'Korea', '2023-07-25', 'Gold'),
  ('C014', '황지안', 'USA', '2023-08-10', NULL),
  ('C015', '안수아', 'Korea', '2023-08-28', 'Bronze');
""")

con.execute("""
CREATE OR REPLACE TABLE orders (
    order_id STRING,
    customer_id STRING,
    order_date DATE,
    status STRING,
    amount DECIMAL(12,2)
);
""")
con.execute("""
INSERT INTO orders VALUES
  ('O0001', 'C001', '2023-09-02', 'Paid', 125000),
  ('O0002', 'C002', '2023-09-05', 'Shipped', 89000),
  ('O0003', 'C001', '2023-09-11', 'Returned', 45000),
  ('O0004', 'C003', '2023-09-15', 'Paid', 230000),
  ('O0005', 'C004', '2023-09-20', 'Cancelled', NULL),
  ('O0006', 'C005', '2023-09-25', 'Shipped', 67000),
  ('O0007', 'C002', '2023-10-01', 'Paid', 158000),
  ('O0008', 'C006', '2023-10-04', 'Placed', 32000),
  ('O0009', 'C007', '2023-10-12', 'Shipped', 410000),
  ('O0010', 'C008', '2023-10-19', 'Paid', 99000),
  ('O0011', 'C001', '2023-10-23', 'Paid', 76000),
  ('O0012', 'C009', '2023-10-28', 'Cancelled', NULL),
  ('O0013', 'C010', '2023-11-02', 'Shipped', 142000),
  ('O0014', 'C004', '2023-11-08', 'Paid', 88000),
  ('O0015', 'C011', '2023-11-13', 'Placed', 53000),
  ('O0016', 'C012', '2023-11-19', 'Shipped', 175000),
  ('O0017', 'C002', '2023-11-24', 'Returned', 61000),
  ('O0018', 'C013', '2023-11-29', 'Paid', 320000),
  ('O0019', 'C005', '2023-12-03', 'Paid', 47000),
  ('O0020', 'C008', '2023-12-09', 'Shipped', 215000),
  ('O0021', 'C014', '2023-12-14', 'Placed', 38000),
  ('O0022', 'C001', '2023-12-20', 'Paid', 134000),
  ('O0023', 'C015', '2023-12-25', 'Shipped', 92000),
  ('O0024', 'C007', '2024-01-03', 'Paid', 268000),
  ('O0025', 'C010', '2024-01-09', 'Cancelled', NULL),
  ('O0026', 'C003', '2024-01-15', 'Paid', 119000),
  ('O0027', 'C013', '2024-01-22', 'Shipped', 405000),
  ('O0028', 'C006', '2024-01-28', 'Paid', 58000),
  ('O0029', 'C004', '2024-02-04', 'Returned', 73000),
  ('O0030', 'C012', '2024-02-11', 'Paid', 187000);
""")

con.execute("""
CREATE OR REPLACE TABLE products (
    product_id STRING,
    product_name STRING,
    category STRING,
    price DECIMAL(12,2)
);
""")
con.execute("""
INSERT INTO products VALUES
  ('P01', '에어러너', 'Running', 89000),
  ('P02', '클래식 스니커즈', 'Sneakers', 65000),
  ('P03', '첼시 부츠', 'Boots', 145000),
  ('P04', '여름 샌들', 'Sandals', 38000),
  ('P05', '트레일 러너', 'Running', 119000),
  ('P06', '캔버스 스니커즈', 'Sneakers', 49000),
  ('P07', '워커 부츠', 'Boots', 175000),
  ('P08', '슬리퍼 샌들', 'Sandals', 25000),
  ('P09', '양말 세트', 'Accessory', 12000),
  ('P10', '운동화 끈', 'Accessory', 5000);
""")

con.execute("""
CREATE OR REPLACE TABLE order_items (
    order_id STRING,
    product_id STRING,
    quantity INT,
    unit_price DECIMAL(12,2)
);
""")
con.execute("""
INSERT INTO order_items VALUES
  ('O0001', 'P01', 1, 125000),
  ('O0002', 'P02', 2, 44500),
  ('O0003', 'P03', 1, 45000),
  ('O0004', 'P04', 2, 115000),
  ('O0006', 'P05', 1, 67000),
  ('O0007', 'P06', 2, 79000),
  ('O0008', 'P07', 1, 32000),
  ('O0009', 'P08', 2, 205000),
  ('O0010', 'P09', 1, 99000),
  ('O0011', 'P01', 2, 38000),
  ('O0013', 'P02', 1, 142000),
  ('O0014', 'P03', 2, 44000),
  ('O0015', 'P04', 1, 53000),
  ('O0016', 'P05', 2, 87500),
  ('O0017', 'P06', 1, 61000),
  ('O0018', 'P07', 2, 160000),
  ('O0019', 'P08', 1, 47000),
  ('O0020', 'P09', 2, 107500),
  ('O0021', 'P01', 1, 38000),
  ('O0022', 'P02', 2, 67000),
  ('O0023', 'P03', 1, 92000),
  ('O0024', 'P04', 2, 134000),
  ('O0026', 'P05', 1, 119000),
  ('O0027', 'P06', 2, 202500),
  ('O0028', 'P07', 1, 58000),
  ('O0029', 'P08', 2, 36500),
  ('O0030', 'P09', 1, 187000);
""")

print("테이블 생성 완료: customers, orders, products, order_items")


테이블 생성 완료: customers, orders, products, order_items


## Q1. 고객 등급(grade)별 매출은 얼마인가요? (3-way INNER JOIN + GROUP BY)


In [1]:
q1 = con.sql("""
SELECT
    c.grade,
    SUM(oi.quantity * oi.unit_price) AS 매출
FROM customers c
JOIN orders o
    ON c.customer_id = o.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
GROUP BY c.grade
ORDER BY 매출 DESC
""").df()
q1


NameError: name 'con' is not defined

> **인사이트**: Silver 등급의 매출이 Gold보다 높게 나타났다 — 등급이 반드시 매출 순서와 일치하지 않는다는 뜻으로, Silver 등급 고객 수나 구매 빈도가 더 많을 가능성이 있다 (원인 확정 아님, 추가 확인 필요).

## Q2. 카테고리별 총매출은 얼마인가요? (INNER JOIN + GROUP BY)

- order_items와 products를 조인해 카테고리별 매출 집계
- **사용**: INNER JOIN, GROUP BY + 집계함수


In [3]:
q2 = con.sql("""
SELECT
    p.category,
    SUM(oi.quantity * oi.unit_price) AS 매출
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
GROUP BY p.category
ORDER BY 매출 DESC
""").df()
q2


,category,매출
0,Sandals,1081000.0
1,Sneakers,989000.0
2,Boots,635000.0
3,Running,600000.0
4,Accessory,501000.0


> **인사이트**: Sandals 카테고리가 개당 가격은 낮은 편(2만~4만원대)인데도 매출 1위를 기록 — 단가보다 판매량(quantity)이 매출을 더 크게 좌우했을 가능성을 시사한다.

## Q3. 카테고리별 베스트셀러 상품은 무엇인가요? (INNER JOIN + FROM 서브쿼리 + 상관 서브쿼리)

- order_items와 products를 조인해 상품별 매출을 먼저 구하고(FROM 서브쿼리)
- 같은 카테고리 안에서 매출이 가장 큰 상품만 남김(상관 서브쿼리)
- **사용**: INNER JOIN, 서브쿼리(FROM절 + 상관 WHERE절)


In [4]:
q3 = con.sql("""
SELECT
    pr.category,
    pr.product_name,
    pr.revenue AS 매출
FROM (
    SELECT
        p.category,
        p.product_name,
        SUM(oi.quantity * oi.unit_price) AS revenue
    FROM order_items oi
    JOIN products p ON oi.product_id = p.product_id
    GROUP BY p.category, p.product_name
) AS pr
WHERE pr.revenue = (
    SELECT MAX(pr2.revenue)
    FROM (
        SELECT
            p2.category,
            p2.product_name,
            SUM(oi2.quantity * oi2.unit_price) AS revenue
        FROM order_items oi2
        JOIN products p2 ON oi2.product_id = p2.product_id
        GROUP BY p2.category, p2.product_name
    ) AS pr2
    WHERE pr2.category = pr.category
)
ORDER BY pr.category
""").df()
q3


,category,product_name,매출
0,Accessory,양말 세트,501000.0
1,Boots,워커 부츠,410000.0
2,Running,트레일 러너,361000.0
3,Sandals,여름 샌들,551000.0
4,Sneakers,캔버스 스니커즈,624000.0


> **인사이트**: 카테고리마다 베스트셀러가 다르게 나타남 (예: Sneakers는 캔버스 스니커즈, Sandals는 여름 샌들). 각 카테고리 내 대표 상품 후보로 활용 가능하나, 표본 기간이 짧아 장기 트렌드로 단정하기는 이르다.

## Q4. 한 번도 주문된 적 없는 상품은 무엇인가요? (LEFT JOIN)

- products를 기준으로 LEFT JOIN하고, 매칭되는 order_items가 없는(NULL) 상품만 필터링
- **사용**: LEFT JOIN


In [5]:
q4 = con.sql("""
SELECT
    p.product_id,
    p.product_name,
    p.category
FROM products p
LEFT JOIN order_items oi
    ON p.product_id = oi.product_id
WHERE oi.product_id IS NULL
""").df()
q4


,product_id,product_name,category
0,P10,운동화 끈,Accessory


> **인사이트**: '운동화 끈'(P10)이 한 번도 판매된 적 없는 것으로 확인됨 — 재고/노출 문제인지 수요 자체가 없는 상품인지는 이 데이터만으로는 알 수 없어 추가 확인이 필요하다.

## Q5. 상품 정가 평균보다 비싸게 팔린 주문 항목은 무엇인가요? (INNER JOIN + WHERE 서브쿼리)

- products 테이블의 정가(price) 평균을 서브쿼리로 구함
- 그 평균보다 실제 판매 단가(unit_price)가 높았던 주문 항목만 필터링
- **사용**: INNER JOIN, 서브쿼리(WHERE절)


In [6]:
q5 = con.sql("""
SELECT
    oi.order_id,
    p.product_name,
    oi.unit_price AS 판매단가,
    p.price AS 정가
FROM order_items oi
JOIN products p
    ON oi.product_id = p.product_id
WHERE oi.unit_price > (
    SELECT AVG(price) FROM products
)
ORDER BY oi.unit_price DESC
""").df()
q5


,order_id,product_name,판매단가,정가
0,O0009,슬리퍼 샌들,205000.0,25000.0
1,O0027,캔버스 스니커즈,202500.0,49000.0
2,O0030,양말 세트,187000.0,12000.0
3,O0018,워커 부츠,160000.0,175000.0
4,O0013,클래식 스니커즈,142000.0,65000.0
5,O0024,여름 샌들,134000.0,38000.0
6,O0001,에어러너,125000.0,89000.0
7,O0026,트레일 러너,119000.0,119000.0
8,O0004,여름 샌들,115000.0,38000.0
9,O0020,양말 세트,107500.0,12000.0


> **인사이트**: 일부 항목(예: 슬리퍼 샌들, 캔버스 스니커즈)은 정가보다 훨씬 높은 단가로 팔린 것으로 나타남 — 단가가 수량 할인/묶음 계산 등으로 정가와 다르게 기록됐을 가능성이 있어, unit_price의 산출 방식을 데이터 담당자에게 확인해볼 필요가 있다 (원인 미확정).

## 종합 요약

| 질문 | JOIN 종류 | 핵심 결과 |
|---|---|---|
| Q1. 등급별 매출 | INNER JOIN ×2 | Silver 등급 매출이 Gold보다 높음 |
| Q2. 카테고리별 매출 | INNER JOIN | Sandals 1위 (단가는 낮은 편) |
| Q3. 카테고리별 베스트셀러 | INNER JOIN + FROM/상관 서브쿼리 | 카테고리마다 대표 상품 상이 |
| Q4. 무판매 상품 | LEFT JOIN | 운동화 끈(P10) 판매 이력 없음 |
| Q5. 정가 평균 초과 판매 | INNER JOIN + WHERE 서브쿼리 | 일부 항목 정가 대비 고가 판매 |

**조건 충족 체크**
- INNER JOIN: Q1, Q2, Q3, Q5 (요구 1회 이상)
- LEFT JOIN: Q4 (요구 1회 이상)
- 서브쿼리: Q3(FROM + 상관 WHERE), Q5(WHERE) (요구 1회 이상)
- JOIN + GROUP BY 집계: Q1, Q2, Q3 (요구 1개 이상)

**참고**: 이번 스키마에는 order_items가 추가되어 orders-products 간 매출/판매량 연결이 가능해졌고,
이전 리포트에서 언급했던 "카테고리별 매출 연결 불가" 한계가 해소되었다.
